In [1]:
pip install nibabel torch torchvision torch-geometric

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install scikit-image

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F
from skimage.transform import resize
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool
from sklearn.model_selection import train_test_split

In [4]:
def load_nifti(file_path):
    img = nib.load(file_path)
    return img.get_fdata()

def preprocess(volume):
    volume = resize(volume, (8,8,8))   # 🔥 reduced size
    volume = (volume - np.min(volume)) / (np.max(volume) - np.min(volume))
    return volume

In [5]:
def create_graph(volume):
    nodes, edges = [], []
    x, y, z = volume.shape

    def idx(i,j,k): return i*y*z + j*z + k

    for i in range(x):
        for j in range(y):
            for k in range(z):
                cur = idx(i,j,k)
                nodes.append([volume[i,j,k]])

                for dx,dy,dz in [(1,0,0),(0,1,0),(0,0,1)]:
                    ni,nj,nk = i+dx, j+dy, k+dz
                    if ni<x and nj<y and nk<z:
                        nb = idx(ni,nj,nk)
                        edges.append([cur, nb])
                        edges.append([nb, cur])

    return Data(
        x=torch.tensor(nodes, dtype=torch.float),
        edge_index=torch.tensor(edges, dtype=torch.long).t().contiguous()
    )

In [6]:
def load_dataset(ad_path, cn_path):
    data_list = []

    # AD = 1
    for f in os.listdir(ad_path):
        if f.endswith(".nii") or f.endswith(".nii.gz"):
            v = preprocess(load_nifti(os.path.join(ad_path,f)))
            g = create_graph(v)
            g.y = torch.tensor([1])
            data_list.append(g)

    # CN = 0
    for f in os.listdir(cn_path):
        if f.endswith(".nii") or f.endswith(".nii.gz"):
            v = preprocess(load_nifti(os.path.join(cn_path,f)))
            g = create_graph(v)
            g.y = torch.tensor([0])
            data_list.append(g)

    return data_list

In [7]:
dataset = load_dataset("alzimers/ad", "alzimers/cn")
print("Total samples:", len(dataset))

Total samples: 30


In [8]:
train_data, test_data = train_test_split(
    dataset,
    test_size=0.2,
    stratify=[d.y.item() for d in dataset],
    random_state=42
)

print("Train:", len(train_data))
print("Test:", len(test_data))

Train: 24
Test: 6


In [9]:
train_loader = DataLoader(train_data, batch_size=4, shuffle=True)
test_loader = DataLoader(test_data, batch_size=4, shuffle=False)

In [13]:
def extract_features(volume):
    import numpy as np
    
    return np.array([
        np.mean(volume),
        np.std(volume),
        np.max(volume),
        np.min(volume)
    ])

In [14]:
features = []
labels = []

for label, path in [(1, "alzimers/ad"), (0, "alzimers/cn")]:
    for f in os.listdir(path):
        if f.endswith(".nii") or f.endswith(".nii.gz"):
            vol = preprocess(load_nifti(os.path.join(path, f)))
            feat = extract_features(vol)

            features.append(feat)
            labels.append(label)

features = np.array(features)
labels = torch.tensor(labels)

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
features = scaler.fit_transform(features)

features = torch.tensor(features, dtype=torch.float)

In [16]:
from sklearn.neighbors import kneighbors_graph

A = kneighbors_graph(features.numpy(), n_neighbors=5, mode='distance')

edge_index = []
edge_weight = []

rows, cols = A.nonzero()

for i, j in zip(rows, cols):
    edge_index.append([i, j])
    edge_weight.append(A[i, j])

edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
edge_weight = torch.tensor(edge_weight, dtype=torch.float)

In [17]:
from torch_geometric.data import Data

graph_data = Data(
    x=features,
    edge_index=edge_index,
    edge_weight=edge_weight,
    y=labels
)

print(graph_data)

Data(x=[30, 4], edge_index=[2, 150], y=[30], edge_weight=[150])


In [18]:
class GNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = GCNConv(4, 16)
        self.conv2 = GCNConv(16, 8)
        self.fc = torch.nn.Linear(8, 2)

    def forward(self, data):
        x, edge_index, edge_weight = data.x, data.edge_index, data.edge_weight

        x = F.relu(self.conv1(x, edge_index, edge_weight))
        x = F.relu(self.conv2(x, edge_index, edge_weight))

        x = self.fc(x)
        return F.log_softmax(x, dim=1)

In [19]:
gnn = GNN()
optimizer = torch.optim.Adam(gnn.parameters(), lr=0.01)

for epoch in range(50):
    gnn.train()
    optimizer.zero_grad()

    out = gnn(graph_data)
    loss = F.nll_loss(out, graph_data.y)

    loss.backward()
    optimizer.step()

    print("Epoch:", epoch, "Loss:", loss.item())

Epoch: 0 Loss: 0.6953796744346619
Epoch: 1 Loss: 0.6909858584403992
Epoch: 2 Loss: 0.6874586343765259
Epoch: 3 Loss: 0.6846333742141724
Epoch: 4 Loss: 0.6825743913650513
Epoch: 5 Loss: 0.680626630783081
Epoch: 6 Loss: 0.6785867214202881
Epoch: 7 Loss: 0.6763505935668945
Epoch: 8 Loss: 0.6738454103469849
Epoch: 9 Loss: 0.6714826226234436
Epoch: 10 Loss: 0.6690206527709961
Epoch: 11 Loss: 0.6666635870933533
Epoch: 12 Loss: 0.6645010709762573
Epoch: 13 Loss: 0.6625747084617615
Epoch: 14 Loss: 0.660541296005249
Epoch: 15 Loss: 0.657880961894989
Epoch: 16 Loss: 0.6551583409309387
Epoch: 17 Loss: 0.6525031328201294
Epoch: 18 Loss: 0.6499166488647461
Epoch: 19 Loss: 0.6472745537757874
Epoch: 20 Loss: 0.6441470980644226
Epoch: 21 Loss: 0.6408933997154236
Epoch: 22 Loss: 0.6375383734703064
Epoch: 23 Loss: 0.6345576047897339
Epoch: 24 Loss: 0.6316462755203247
Epoch: 25 Loss: 0.6282151341438293
Epoch: 26 Loss: 0.6245633363723755
Epoch: 27 Loss: 0.6212993264198303
Epoch: 28 Loss: 0.618107914924621

In [20]:
gnn.eval()
pred = gnn(graph_data).argmax(dim=1)

acc = (pred == labels).sum().item() / len(labels)
print("Accuracy:", acc)

Accuracy: 0.7333333333333333
